# Brent · `lpa_app_no` (web: "LPA Number")

**Goal:** understand the data, so that a chat bot can answer any question about it
more easily and correctly.

Brent only (golden rule): every query carries the location-based Brent filter.
Data comes from the **live API**.

Workflow: add a step here → discuss the findings → decide whether it belongs in
`knowledge/brent.md` → add it.

In [1]:
from pathlib import Path

import httpx
import pandas as pd
from dotenv import dotenv_values

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
env = dotenv_values(PROJECT_ROOT / ".env")

BASE_URL = "https://planningdata.london.gov.uk/api-guest"
HEADERS = {"X-API-AllowRequest": env["DATAHUB_ALLOW_REQUEST"]}


def brent_filter() -> dict:
    # Location-based scope: borough is Brent, or borough is missing and Brent decided it.
    return {
        "bool": {
            "should": [
                {"term": {"borough.raw": "Brent"}},
                {"bool": {
                    "filter": [{"term": {"lpa_name.raw": "Brent"}}],
                    "must_not": [{"exists": {"field": "borough.raw"}}],
                }},
            ],
            "minimum_should_match": 1,
        }
    }


def search(body: dict) -> dict:
    response = httpx.post(f"{BASE_URL}/applications/_search", headers=HEADERS, json=body, timeout=60)
    response.raise_for_status()
    return response.json()

## Step 1 · Nulls and empty values

Fetch `lpa_app_no` for every Brent application (paged with `search_after`, sorted by
`id`, because there are more than 10,000). Then count:
- **null**: no value at all;
- **empty or blank**: a string with nothing in it, or only whitespace.

In [2]:
rows, after = [], None
while True:
    body = {"size": 5000, "query": brent_filter(), "_source": ["id", "lpa_app_no"], "sort": [{"id": "asc"}]}
    if after:
        body["search_after"] = after
    page = search(body)["hits"]["hits"]
    if not page:
        break
    rows += [h["_source"] for h in page]
    after = page[-1]["sort"]

df = pd.DataFrame(rows).reindex(columns=["id", "lpa_app_no"])
print("Brent applications:", len(df))

Brent applications: 36952


In [3]:
value = df["lpa_app_no"]
is_null = value.isna()
is_blank = value.notna() & (value.astype(str).str.strip() == "")

summary = pd.Series({
    "null": int(is_null.sum()),
    "empty or blank": int(is_blank.sum()),
    "filled": int((~is_null & ~is_blank).sum()),
})
summary["total"] = summary.sum()
summary.to_frame("count").assign(**{"share_%": lambda t: (100 * t["count"] / len(df)).round(2)})

,count,share_%
null,0,0.0
empty or blank,0,0.0
filled,36952,100.0
total,36952,100.0


### Findings · Step 1

Live API, 2026-11-10, 36,952 Brent applications:

| | Count | Share |
|---|---|---|
| null | 0 | 0% |
| empty or blank | 0 | 0% |
| filled | 36,952 | 100% |

- **Every record has an `lpa_app_no`**, including the 407 empty records.

## Step 2 · Do the values match the schema type?

Count the Python type of every value in `df`: `str` = JSON string, `int` / `float` =
JSON number, `NoneType` = null.

In [4]:
types = df["lpa_app_no"].map(lambda v: "NoneType" if v is None or v != v else type(v).__name__)
display(types.value_counts().to_frame("count"))

not_str = df[~types.isin(["str", "NoneType"])]
print("values that are not strings:", len(not_str))
not_str["lpa_app_no"].value_counts().head(30).to_frame("count")

,count
lpa_app_no,
str,36952


values that are not strings: 0


,count
lpa_app_no,


### Findings · Step 2

Live API, 2026-11-10:

| Python type | Count |
|---|---|
| `str` | 36,952 |
| number | 0 |

- **Every value is a JSON string**, as the schema says (`string`). No type mismatch.

## Step 3 · Most common values

For a reference number, "most common" means duplicates: count identical `lpa_app_no`
values in `df` and show every value used more than once, with the ids. `shown_as` uses
`repr()` so hidden characters are visible.

In [5]:
counts = df["lpa_app_no"].value_counts()
print("distinct:", len(counts), "| filled:", df["lpa_app_no"].notna().sum(),
      "| used more than once:", int((counts > 1).sum()), "values,", int(counts[counts > 1].sum()), "records")

dupes = df[df["lpa_app_no"].isin(counts[counts > 1].index)].sort_values(["lpa_app_no", "id"])
dupes = dupes.assign(shown_as=dupes["lpa_app_no"].map(repr))
dupes.groupby("lpa_app_no").agg(count=("id", "size"), ids=("id", list)).sort_values("count", ascending=False).head(30)

distinct: 36948 | filled: 36952 | used more than once: 4 values, 8 records


,count,ids
lpa_app_no,,
18/1022,2,"[Brent-18_1022, OPDC-18_1022]"
18/2581,2,"[Brent-18_2581, OPDC-18_2581]"
19/2236,2,"[Brent-19_2236, OPDC-19_2236]"
19/2890,2,"[Brent-19_2890, OPDC-19_2890]"


### Findings · Step 3

Live API, 2026-11-10: 36,952 values, 36,948 distinct.

- **Unique apart from 4 values**, each used twice:

| `lpa_app_no` | ids |
|---|---|
| 18/1022 | `Brent-18_1022`, `OPDC-18_1022` |
| 18/2581 | `Brent-18_2581`, `OPDC-18_2581` |
| 19/2236 | `Brent-19_2236`, `OPDC-19_2236` |
| 19/2890 | `Brent-19_2890`, `OPDC-19_2890` |

- These are the 4 applications recorded under both Brent and OPDC (open question 12).
  No other reference number repeats.
- The `id` is the authority name plus the reference (`Brent-18_1022`), so `id` is unique
  even where `lpa_app_no` is not.

## Step 4 · The 4 Brent / OPDC pairs side by side

Fetch both records of each pair with the web columns plus `lpa_name`, `bo_system` and
`last_updated`, and show them next to each other. `filled` counts the non-empty fields
per record.

In [6]:
COLUMNS = [
    "id", "lpa_app_no", "lpa_name", "borough", "valid_date", "status", "decision", "decision_date",
    "application_type", "description", "site_name", "site_number", "street_name", "locality", "postcode",
    "decision_target_date", "appeal_decision", "appeal_decision_date", "url_planning_app", "bo_system", "last_updated",
]
pair_ids = dupes["id"].tolist()
hits = search({"size": 20, "query": {"bool": {"filter": [brent_filter(), {"ids": {"values": pair_ids}}]}},
               "_source": COLUMNS})["hits"]["hits"]
pairs = pd.DataFrame([h["_source"] for h in hits]).reindex(columns=COLUMNS).sort_values(["lpa_app_no", "id"])
pairs["filled"] = pairs[COLUMNS].notna().sum(axis=1)
pairs["description"] = pairs["description"].str[:80]
pd.set_option("display.max_colwidth", 80)
pairs.set_index("id").T

id,Brent-18_1022,OPDC-18_1022,Brent-18_2581,OPDC-18_2581,Brent-19_2236,OPDC-19_2236,Brent-19_2890,OPDC-19_2890
lpa_app_no,18/1022,18/1022,18/2581,18/2581,19/2236,19/2236,19/2890,19/2890
lpa_name,Brent,OPDC,Brent,OPDC,Brent,OPDC,Brent,OPDC
borough,Brent,Brent,Brent,Brent,Brent,Brent,Brent,Brent
valid_date,12/04/2018,NaN,07/09/2018,NaN,21/06/2019,NaN,14/08/2019,NaN
status,Approved,Lapsed,Lapsed,Lapsed,Approved,Lapsed,Approved,Lapsed
decision,Approved,Approved,Approved,Approved,Approved,Approved,Approved,Approved
decision_date,07/06/2018,07/06/2018,20/05/2020,20/05/2020,15/08/2019,15/08/2019,09/10/2019,09/10/2019
application_type,All Other,All Other,All Other,All Other,All Other,All Other,All Other,All Other
description,Installation of two levels of storage mezzanine totalling 3256m2.,Installation of two levels of storage mezzanine totalling 3256m2.,Construction of part fifth and sixth storey (part fourth floor and part fift...,Construction of part fifth and sixth storey (part fourth floor and part fift...,"Proposed flexible use of site for business, general industry and storage and...","Proposed flexible use of site for business, general industry and storage and...",Change of use of industrial estate (units 1 to 16) from general industry (Us...,Change of use of industrial estate (units 1 to 16) from general industry (Us...
site_name,"Unit 8, Central Park, Central Way, London, NW10 7FY",Central Park,"Victoria Care Centre, Acton Lane, London, NW10 7BR",Victoria Care Centre,"3 Willen Field Road, London, NW10 7BQ",3,"All units, Townsend Industrial Estate, Waxlow Road, London, NW10 7NU",Townsend Industrial Estate


### Findings · Step 4

Live API, 2026-11-10, the 4 pairs:

| | Brent copy | OPDC copy |
|---|---|---|
| `lpa_app_no`, `description`, `decision`, `decision_date`, `application_type`, `borough` | same | same |
| `lpa_name` | Brent | OPDC |
| `valid_date`, `decision_target_date` | filled | missing |
| `url_planning_app` | Brent planning portal | missing |
| address | full address in `site_name` | split over `site_name` / `site_number` / `street_name` / `postcode` |
| `status` | Approved (3), Lapsed (18/2581) | Lapsed (all 4) |
| `bo_system` | CSV Data Import | FME Legacy Data Import (3), Portal (19/2236) |
| `last_updated` | 2026-11-05 | 2025-08 / 2025-10 |

- **Same application twice:** same reference, description, decision and decision date.
- **The Brent copy has more:** valid date and target date (needed for "decided on time?"
  and trends), the planning portal URL, and it is updated with the regular feed. The OPDC
  copy has nothing the Brent copy lacks: its address parts give the same address.
- **Two differences the data can't settle:** `status` (Brent says Approved, OPDC says
  Lapsed for 3 pairs) and `lpa_name` (which authority decided). The sites are all in NW10 7
  (Park Royal).